# `LayerNorm`

参考：[LayerNorm](https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html)

{class}`torch.nn.LayerNorm` 是 **PyTorch** 中实现 **层归一化** （Layer Normalization）的模块，用来对神经网络输入的特征维度进行标准化处理，从而加速训练、稳定模型表现，并减少内部协变量偏移（internal covariate shift）。 

```{admonition} 🧠 核心思想
- **归一化范围**：对 **每个样本** 的指定特征维度（`normalized_shape`）计算均值和方差，而不是像 BatchNorm 那样在整个 batch 上计算。
- **计算公式**：

    $$
    y = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \times \gamma + \beta
    $$

  - $\mu$：特征均值  
  - $\sigma$：特征标准差  
  - $\epsilon$：防止除零的小常数（默认 `1e-5`）  
  - $\gamma$、$\beta$：可学习的缩放和偏移参数（`elementwise_affine=True` 时存在）
```

```{admonition} 🔍 与 BatchNorm 的区别
| 特性 | LayerNorm | BatchNorm |
|------|----------|-----------|
| 归一化维度 | 每个样本的特征维度 | 整个 batch 的同一特征 |
| 对 batch size 敏感性 | 不敏感 | 敏感 |
| 适用场景 | RNN、Transformer、小批量或变长序列 | CNN、大批量训练 |
| 统计量计算 | 样本内计算 | batch 内计算 |
```

```{admonition} ⚙️ 常用参数
- **`normalized_shape`**：要归一化的维度大小（int 或 tuple）
- **`eps`**：数值稳定性常数（默认 `1e-5`）
- **`elementwise_affine`**：是否使用可学习的 $\gamma$ 和 $\beta$（默认 `True`）
- **`bias`**：是否学习偏置（仅在 `elementwise_affine=True` 时有效）
```
```{admonition} 🚀 适用场景
- **Transformer**（如 BERT、GPT 系列）
- **RNN/LSTM/GRU** 等顺序模型
- 小批量训练或变长输入序列
- 对 batch size 不稳定的任务
```

## 💡 使用示例

In [1]:
import torch
import torch.nn as nn

# ===== NLP 示例：对 embedding 维度做 LayerNorm =====
batch_size = 4
seq_len = 6
embed_dim = 8

# 模拟一个批次的词向量输入
x_nlp = torch.randn(batch_size, seq_len, embed_dim)

# 创建 LayerNorm，归一化最后一维（embedding 维度）
layer_norm_nlp = nn.LayerNorm(normalized_shape=embed_dim)

# 前向计算
out_nlp = layer_norm_nlp(x_nlp)

print("=== NLP 示例 ===")
print("输入形状:", x_nlp.shape)
print("输出形状:", out_nlp.shape)
print("输出均值:", out_nlp.mean(dim=-1))  # 每个 token 的均值接近 0
print("输出方差:", out_nlp.var(dim=-1, unbiased=False))  # 方差接近 1


# ===== 图像示例：对 C,H,W 维度做 LayerNorm =====
N, C, H, W = 2, 3, 4, 4
x_img = torch.randn(N, C, H, W)

# 创建 LayerNorm，归一化通道+空间维度
layer_norm_img = nn.LayerNorm(normalized_shape=[C, H, W])

out_img = layer_norm_img(x_img)

print("\n=== 图像示例 ===")
print("输入形状:", x_img.shape)
print("输出形状:", out_img.shape)
print("单个样本均值:", out_img[0].mean())  # 接近 0
print("单个样本方差:", out_img[0].var(unbiased=False))  # 接近 1

=== NLP 示例 ===
输入形状: torch.Size([4, 6, 8])
输出形状: torch.Size([4, 6, 8])
输出均值: tensor([[-3.7253e-09,  1.4901e-08, -2.9802e-08,  1.8626e-09, -2.2352e-08,
          5.0291e-08],
        [ 4.4703e-08,  4.4703e-08, -1.4203e-08, -3.7253e-08, -4.4703e-08,
          0.0000e+00],
        [-1.4901e-08,  2.9802e-08,  1.4901e-08, -1.4901e-08, -2.2352e-08,
          4.9360e-08],
        [-3.3528e-08, -2.9802e-08,  2.9802e-08,  2.2352e-08, -4.2841e-08,
          1.4901e-08]], grad_fn=<MeanBackward1>)
输出方差: tensor([[1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000]],
       grad_fn=<VarBackward0>)

=== 图像示例 ===
输入形状: torch.Size([2, 3, 4, 4])
输出形状: torch.Size([2, 3, 4, 4])
单个样本均值: tensor(0., grad_fn=<MeanBackward0>)
单个样本方差: tensor(1.0000, grad_fn=<VarBackward0>)


```{admonition} 🔍 运行结果特点
- **NLP 示例**：每个 token 的 embedding 经过归一化后，均值接近 0，方差接近 1。
- **图像示例**：每张图片的 `(C, H, W)` 整体被归一化，适合小 batch 或 Transformer 结构中的视觉任务。
- `normalized_shape` 决定了归一化的维度范围。
- `elementwise_affine=True`（默认）时，LayerNorm 会学习缩放参数 $\gamma$ 和 $\beta$。
```

## LayerNorm 工作原理

LayerNorm 的目标是 **让每个样本的特征分布更稳定**，从而加快训练、减少梯度震荡。  
它的关键点是：  
- **归一化范围**：对 **单个样本** 的指定特征维度（`normalized_shape`）计算均值和方差，而不是像 BatchNorm 那样在整个 batch 上计算。  
- **不依赖 batch size**：即使 batch size 很小（甚至为 1），它依然能稳定工作，非常适合 RNN、Transformer 等结构。  


### 🔬 计算流程
假设输入是一个样本的特征向量 $ x $：  

1. **计算均值**

   $$
   \mu = \frac{1}{H} \sum_{i=1}^H x_i
   $$

   这里 $H$ 是归一化的特征维度大小。

2. **计算方差**

   $$
   \sigma^2 = \frac{1}{H} \sum_{i=1}^H (x_i - \mu)^2
   $$

3. **归一化**

   $$
   \hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}}
   $$

   其中 $\epsilon$ 是防止除零的小常数（默认 $1e{-5}$）。

4. **缩放和平移（可学习参数）** 

   $$
   y_i = \gamma \cdot \hat{x}_i + \beta
   $$  

   - $\gamma$：缩放系数  
   - $\beta$：偏移系数  
   - 这两个参数在训练中会被学习到，使模型在归一化后仍能恢复必要的分布特征。

### 💡 直观理解
你可以把 LayerNorm 想象成：  
> “对每个样本单独做一次特征标准化，让它的特征分布更平滑、更稳定，然后再让模型自己学会如何调整回适合任务的分布。”

这就像在做菜前，先把每份食材的味道调到一个统一的基准，再根据菜谱加上不同的调料（$\gamma$ 和 $\beta$）来做出不同的风味。


## LayerNorm 的优势  

```{admonition} 🌟 1. 对 batch size 不敏感  
- **BatchNorm** 在小批量训练时，统计量（均值、方差）容易不稳定，导致模型性能下降。  
- **LayerNorm** 是在 **单个样本内部** 计算均值和方差，不依赖 batch 维度，所以即使 batch size = 1 也能稳定工作。  
- 这对 **在线推理**、**小数据集训练**、**变长序列** 特别有用。  
```

```{admonition} 🔄 2. 适合序列建模（RNN / Transformer）  
- 在 NLP、语音等任务中，输入通常是 **变长序列**，BatchNorm 很难直接应用。  
- LayerNorm 对每个时间步的特征做归一化，不会破坏时间顺序信息，非常适合 **RNN、LSTM、GRU** 以及 **Transformer**。  
```

```{admonition} ⚖️ 3. 稳定训练过程  
- 通过让每个样本的特征分布均值接近 0、方差接近 1，减少了 **内部协变量偏移**（Internal Covariate Shift）。  
- 这能让梯度更稳定，学习率可以设得更高，收敛速度更快。  
```

```{admonition} 🎯 4. 提高泛化能力  
- 归一化减少了特征分布的波动，使模型更容易学习到稳定的模式，从而在测试集上表现更好。  
```

```{admonition} 🛠 5. 易于集成  
- 在 PyTorch 中只需一行 `nn.LayerNorm(normalized_shape)` 就能使用。  
- 可与 **残差连接（Residual Connection）** 搭配，进一步提升深层网络的稳定性（Transformer 就是这样做的）。  
```

```{admonition} ✅ **总结**：  
LayerNorm 的最大好处是 **稳定性 + 通用性** —— 它不依赖 batch 统计量，适合各种输入形状和任务场景，尤其是在 **小批量、变长序列、深层网络** 中表现突出。  
```

## LayerNorm 应用案例  

---

### 📚 1. Transformer 系列模型（BERT / GPT / ViT）
- **位置**：出现在 **多头注意力层** 和 **前馈网络层** 之后，通常与残差连接（Residual Connection）一起使用。
- **作用**：稳定深层网络的训练，防止梯度爆炸/消失。
- **例子**：BERT 的每个 Encoder Block 都是：
  ```
  x = x + MultiHeadAttention(...)
  x = LayerNorm(x)
  x = x + FeedForward(...)
  x = LayerNorm(x)
  ```
- **好处**：即使网络有几十层，训练依然稳定。

---

### 🗣 2. RNN / LSTM / GRU 等序列模型
- **问题**：BatchNorm 在变长序列或 batch size 很小时效果差。
- **解决**：LayerNorm 对每个时间步的隐藏状态单独归一化，不依赖 batch 统计量。
- **例子**：在语音识别、机器翻译等任务中，将 LayerNorm 应用于 LSTM 的输入和隐藏状态，显著提升收敛速度和泛化能力。

---

### 🖼 3. 小批量或在线推理的视觉任务
- **问题**：推理时 batch size 可能为 1，BatchNorm 的统计量不稳定。
- **解决**：LayerNorm 在 **单张图片** 的通道+空间维度上归一化。
- **例子**：Vision Transformer (ViT) 在图像分类中完全用 LayerNorm 取代 BatchNorm，适配小 batch 训练。

---

### 🎯 4. 强化学习（Reinforcement Learning）
- **场景**：在策略网络（Policy Network）或价值网络（Value Network）中使用 LayerNorm。
- **好处**：减少输入特征分布的波动，使训练过程更稳定，尤其在环境状态变化较大的任务中。

---

### 💡 5. 生成模型（GAN / Diffusion）
- 在生成器或判别器中使用 LayerNorm，可以在小数据集或不稳定训练条件下保持梯度平稳，避免模式崩溃。

---

✅ **总结**  
LayerNorm 的核心优势是 **不依赖 batch 统计量**，因此在 **小批量、变长序列、深层网络** 中表现突出。它几乎是 Transformer 架构的“标配”，也是 RNN、ViT 等模型稳定训练的关键组件。  

---